In [1]:
"""
МИНИ-GPT с настоящим BPE-токенизатором
=========================================
Исправленная версия: теперь модель реально использует обученный
токенизатор (а не путь к файлу как "текст").
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

torch.manual_seed(42)

# ---------------------------------------------------------------
# 1. ТОКЕНИЗАТОР: обучаем BPE на своём файле
# ---------------------------------------------------------------

files = ["/kaggle/input/datasets/pokilondron1/datasetst/dataset.txt"]  # ЗАМЕНИ на свой путь к файлу

tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = Whitespace()

# ВАЖНО: vocab_size должен быть разумным относительно размера текста.
# Для маленького файла (как тестовый) хватит 300-500.
# Для целой книги можно 8000-16000. 30000 имеет смысл только для
# ОЧЕНЬ большого корпуса текста (десятки мегабайт).
VOCAB_SIZE = 500

trainer = BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=["<unk>", "<s>", "</s>"])
tokenizer.train(files, trainer)
tokenizer.save("my_tokenizer.json")

# ---------------------------------------------------------------
# 2. ДАННЫЕ: читаем ТЕКСТ (а не путь к файлу!) и кодируем через tokenizer
# ---------------------------------------------------------------

# читаем содержимое файла как обычный текст
with open(files[0], "r", encoding="utf-8") as f:
    text = f.read()

vocab_size = tokenizer.get_vocab_size()   # берём реальный размер словаря токенизатора

def encode(s):
    return tokenizer.encode(s).ids        # текст -> список id токенов

def decode(ids):
    return tokenizer.decode(ids)          # список id токенов -> текст

data = torch.tensor(encode(text), dtype=torch.long)
print(f"Длина текста в токенах: {len(data)}")
print(f"Размер словаря: {vocab_size}")

# ---------------------------------------------------------------
# ГИПЕРПАРАМЕТРЫ (уменьшены до разумного "лёгкого" размера)
# ---------------------------------------------------------------

block_size = 128      # Увеличиваем контекст в 2 раза, чтобы она видела структуру предложений!
batch_size = 32       # Немного снижаем батч, чтобы компенсировать память под контекст
n_embed = 192         # Оптимальная скрытая размерность
n_heads = 6           # 192 / 6 = 32 (идеальный размер для одной головы внимания)
n_layers = 4          # Сокращаем слои до 4. Этого с головой хватит для символьной модели!
dropout = 0.2         # 0.5 — слишком жесткий дропаут для такой маленькой сети, зануляет слишком много инфы
learning_rate = 5e-4  # Чуть-чуть поднимаем шаг, так как модель стала меньше и маневреннее
steps = 3000          # Ставим 3000 шагов (благодаря легкой архитектуре они пролетят мгновенно)

device = "cuda" if torch.cuda.is_available() else "cpu"

def get_batch():
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


# ---------------------------------------------------------------
# 3. АРХИТЕКТУРА (без изменений — тут всё было правильно)
# ---------------------------------------------------------------

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embed, n_heads):
        super().__init__()
        head_size = n_embed // n_heads
        self.sa = MultiHeadAttention(n_heads, head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embed)
        self.position_embedding = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(*[Block(n_embed, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx


# ---------------------------------------------------------------
# 4. ОБУЧЕНИЕ
# ---------------------------------------------------------------

model = MiniGPT().to(device)
print(f"\nПараметров в модели: {sum(p.numel() for p in model.parameters())}")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("\n--- Обучение ---")
for step in range(steps):
    xb, yb = get_batch()
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 50 == 0:
        print(f"шаг {step}: loss = {loss.item():.4f}")

print(f"финальный loss = {loss.item():.4f}")

# ---------------------------------------------------------------
# 5. ГЕНЕРАЦИЯ ТЕКСТА
# ---------------------------------------------------------------

print("\n--- Генерация текста ---")

# ВАЖНО: нельзя просто брать text[:1] (первый СИМВОЛ) — если это пробел
# или перенос строки, BPE-токенизатор с Whitespace pre-tokenizer'ом
# выбросит его и вернёт ПУСТОЙ список токенов -> модель падает с IndexError,
# т.к. не может взять logits[:, -1, :] у тензора с 0 токенов.
#
# Решение: берём стартовый токен из уже закодированных данных (data),
# там гарантированно есть хотя бы один настоящий токен.
start_idx = data[:1].unsqueeze(0).to(device)  # (1, 1) - первый токен из data, а не из сырого текста

generated = model.generate(start_idx, max_new_tokens=100)
print(decode(generated[0].tolist()))




Длина текста в токенах: 21593
Размер словаря: 500

Параметров в модели: 1994612

--- Обучение ---
шаг 0: loss = 6.4018
шаг 50: loss = 5.3556
шаг 100: loss = 4.8477
шаг 150: loss = 4.4204
шаг 200: loss = 4.1661
шаг 250: loss = 4.0026
шаг 300: loss = 3.7724
шаг 350: loss = 3.7466
шаг 400: loss = 3.7096
шаг 450: loss = 3.5632
шаг 500: loss = 3.4894
шаг 550: loss = 3.2949
шаг 600: loss = 3.2156
шаг 650: loss = 3.0006
шаг 700: loss = 2.9062
шаг 750: loss = 2.7214
шаг 800: loss = 2.4965
шаг 850: loss = 2.2996
шаг 900: loss = 2.1101
шаг 950: loss = 2.0674
шаг 1000: loss = 1.8153
шаг 1050: loss = 1.6991
шаг 1100: loss = 1.5362
шаг 1150: loss = 1.4184
шаг 1200: loss = 1.3295
шаг 1250: loss = 1.2274
шаг 1300: loss = 1.1110
шаг 1350: loss = 1.0181
шаг 1400: loss = 0.9191
шаг 1450: loss = 0.8881
шаг 1500: loss = 0.8535
шаг 1550: loss = 0.8269
шаг 1600: loss = 0.7753
шаг 1650: loss = 0.6757
шаг 1700: loss = 0.6818
шаг 1750: loss = 0.6454
шаг 1800: loss = 0.6089
шаг 1850: loss = 0.6026
шаг 1900: 

In [2]:
pip install tokenizers

Note: you may need to restart the kernel to use updated packages.
